# Product Evidence Platform — Azure ML Notebook Runner

This is the only supported notebook.

Fresh-clone setup is intentionally one command after `.env` is edited:

```bash
cp .env.example .env
# replace the SerpAPI and LLM placeholders
./scripts/azureml_startup.sh
```

The startup script builds and recreates Docker Compose, handles Azure ML `cloudfiles` permission limitations automatically, waits for strict health, writes `data/runtime/stack_health.json`, and prints the available feature-set names.

The runtime performs exactly three SerpAPI searches and gives every retained candidate an isolated LLM-controlled browser investigation:

`observe → plan → safe browser action → observe → repeat`

The LLM controls investigation strategy. Deterministic code still enforces exact product identity, browser access, feature completeness, conflicts, scrapability, scope priority, and durable `primary_url` acceptance.

`COMPLETED` and `REVIEW_REQUIRED` are successful terminal workflow states. `REVIEW_REQUIRED` means execution completed but no investigated URL passed every mandatory gate. Only `FAILED` represents an execution failure.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import time
from pathlib import Path
from pprint import pprint
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "docker-compose.yml").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root containing docker-compose.yml")


PROJECT_ROOT = find_project_root()
AGENT_URL = os.getenv("PRODUCT_AGENT_URL", "http://127.0.0.1:8788").rstrip("/")
POLL_SECONDS = 3
HEARTBEAT_SECONDS = 30
TERMINAL_STATUSES = {"COMPLETED", "REVIEW_REQUIRED", "FAILED"}
BOOTSTRAP_STATUS_PATH = PROJECT_ROOT / "data" / "runtime" / "stack_health.json"


def api_json(method: str, path: str, payload: dict | None = None, timeout: int = 30) -> dict:
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{AGENT_URL}{path}",
        data=body,
        method=method,
        headers={"Content-Type": "application/json"},
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Agent API returned HTTP {exc.code}: {detail}\n"
            "Run ./scripts/azureml_startup.sh from the repository root and rerun this cell."
        ) from exc
    except URLError as exc:
        raise RuntimeError(
            f"Cannot reach {AGENT_URL}. Run ./scripts/azureml_startup.sh first."
        ) from exc


def available_feature_sets() -> list[str]:
    private_root = PROJECT_ROOT / "inputs" / "private"
    return sorted(path.stem for path in private_root.glob("*.json") if path.is_file())


def read_bootstrap_status() -> dict:
    if not BOOTSTRAP_STATUS_PATH.is_file():
        return {}
    return json.loads(BOOTSTRAP_STATUS_PATH.read_text(encoding="utf-8"))


def check_health() -> dict:
    health = api_json("GET", "/health", timeout=15)
    if health.get("status") != "healthy":
        raise RuntimeError(f"Platform is not healthy: {json.dumps(health, indent=2)}")
    configuration = health.get("configuration") or {}
    if not configuration.get("three_stage_contract_enforced"):
        raise RuntimeError("Agent is not running the strict three-stage contract")
    if configuration.get("serpapi_request_limit") != 3:
        raise RuntimeError("Agent search credit limit is not exactly 3")
    if not configuration.get("agentic_browser_contract_enforced"):
        raise RuntimeError("Agent is not running the required LLM-agentic browser contract")
    if not configuration.get("llm_configured"):
        raise RuntimeError("LLM configuration is not valid")
    browser = health.get("browser_service") or {}
    if not browser.get("agentic_tools"):
        raise RuntimeError("Browser service does not expose agentic session tools")
    return health


def submit_product(product: dict, feature_set: str) -> str:
    response = api_json("POST", "/v1/jobs", {"product": product, "feature_set": feature_set})
    return response["job_id"]


def wait_for_job(
    job_id: str,
    poll_seconds: int = POLL_SECONDS,
    heartbeat_seconds: int = HEARTBEAT_SECONDS,
) -> dict:
    started = time.monotonic()
    last_signature = None
    last_printed_at = 0.0
    while True:
        status = api_json("GET", f"/v1/jobs/{job_id}", timeout=15)
        signature = (
            status.get("status"),
            status.get("stage"),
            status.get("message"),
        )
        now = time.monotonic()
        elapsed = int(now - started)
        should_print = signature != last_signature or now - last_printed_at >= heartbeat_seconds
        if should_print:
            line = (
                f"{job_id}: {status['status']} | "
                f"{status.get('stage', '')} | {status.get('message', '')}"
            )
            if signature == last_signature:
                line += f" | still running ({elapsed}s elapsed)"
            print(line)
            last_signature = signature
            last_printed_at = now
        if status["status"] in TERMINAL_STATUSES:
            if status["status"] == "FAILED":
                raise RuntimeError(status.get("error") or status.get("message") or "Job failed")
            return status
        time.sleep(poll_seconds)


def run_product(product: dict, feature_set: str) -> dict:
    job_id = submit_product(product, feature_set)
    wait_for_job(job_id)
    return api_json("GET", f"/v1/jobs/{job_id}/result", timeout=60)


def host_artifact_dir(result: dict) -> Path | None:
    row_id = (result.get("product") or {}).get("row_id")
    return PROJECT_ROOT / "data" / "artifacts" / row_id if row_id else None


def summarize_result(result: dict) -> dict:
    product = result.get("product") or {}
    search = result.get("search") or {}
    agentic = result.get("agentic_browser") or {}
    product_match = result.get("product_match") or {}
    acceptance = result.get("primary_url_acceptance") or {}
    evidence_set = result.get("evidence_set") or {}
    host_dir = host_artifact_dir(result)
    return {
        "row_id": product.get("row_id"),
        "job_status": result.get("job_status"),
        "coding_ready": result.get("coding_ready"),
        "primary_url": result.get("primary_url"),
        "supplementary_urls": result.get("supplementary_urls") or [],
        "serpapi_requests_used": search.get("serpapi_requests_used"),
        "search_stages": [stage.get("name") for stage in search.get("stages") or []],
        "agentic_browser_enabled": agentic.get("enabled"),
        "candidate_urls_admitted": agentic.get("candidate_urls_admitted"),
        "candidate_investigations_completed": agentic.get("candidate_investigations_completed"),
        "candidate_investigations_failed": agentic.get("candidate_investigations_failed"),
        "selection_scope": product_match.get("selection_scope"),
        "url_decision_status": product_match.get("url_decision_status"),
        "strict_primary_url_accepted": acceptance.get("accepted"),
        "acceptance_reasons": acceptance.get("reasons") or [],
        "evidence_status": evidence_set.get("status"),
        "coverage": evidence_set.get("total_coverage"),
        "missing_features": evidence_set.get("missing_features") or [],
        "container_artifact_dir": result.get("artifact_dir"),
        "host_artifact_dir": str(host_dir) if host_dir else None,
    }


bootstrap_status = read_bootstrap_status()
health = check_health()
feature_sets = available_feature_sets()
print(f"Repository root: {PROJECT_ROOT}")
print(f"Agent URL: {AGENT_URL}")
print(f"Bootstrap status file: {BOOTSTRAP_STATUS_PATH}")
print(f"Available FEATURE_SET values: {feature_sets}")
if not feature_sets:
    raise RuntimeError("No feature schema exists. Rerun ./scripts/azureml_startup.sh")
pprint(health)


## Single product

Replace the product values, select one of the feature sets printed above, and set `RUN_SINGLE_PRODUCT = True`.

`main_text` and `country_code` are mandatory. `retailer_name`, `ean`, and `language_code` are optional. Keep EAN/GTIN as text.


In [ ]:
FEATURE_SET = feature_sets[0]
RUN_SINGLE_PRODUCT = False

product = {
    "row_id": "ROW-001",
    "main_text": "Replace with the exact product identity text",
    "country_code": "CH",
    "retailer_name": None,
    "ean": None,
    "language_code": None,
}

if RUN_SINGLE_PRODUCT:
    if product["main_text"].startswith("Replace with"):
        raise ValueError("Replace product['main_text'] before running")
    result = run_product(product, FEATURE_SET)
    pprint(summarize_result(result))
else:
    print("Ready. Replace the product input and set RUN_SINGLE_PRODUCT = True.")


## Inspect the LLM-controlled candidate investigations

Run after a product result exists.


In [ ]:
agentic = result.get("agentic_browser") or {}
investigations = result.get("candidate_investigations") or []

pprint(agentic)
for investigation in investigations:
    print(
        investigation.get("candidate_id"),
        investigation.get("status"),
        investigation.get("termination_reason"),
        f"turns={investigation.get('turns_used')}",
        f"actions={investigation.get('actions_executed')}",
    )
    for turn, plan in enumerate(investigation.get("plans") or [], start=1):
        print(
            f"  turn {turn}:",
            plan.get("action"),
            plan.get("element_id") or plan.get("direction") or "",
            "-",
            plan.get("reason"),
        )


## Inspect deterministic final acceptance and evidence


In [ ]:
pprint(result.get("search") or {})
pprint(result.get("product_match") or {})
pprint(result.get("primary_url_acceptance") or {})
pprint(result.get("evidence_set") or {})
pprint(result.get("feature_assessments") or [])
pprint(result.get("browser_evidence") or [])

artifact_path = host_artifact_dir(result)
if artifact_path and artifact_path.is_dir():
    print(f"Host artifact directory: {artifact_path}")
    for path in sorted(artifact_path.glob("CAND-*/agentic/investigation.json")):
        print(f"\n--- {path.relative_to(artifact_path)} ---")
        pprint(json.loads(path.read_text(encoding="utf-8")))
    acceptance_path = artifact_path / "primary_url_acceptance.json"
    if acceptance_path.is_file():
        print("\n--- primary_url_acceptance.json ---")
        pprint(json.loads(acceptance_path.read_text(encoding="utf-8")))
else:
    print("No repository-local artifact directory was found for this result.")


## Optional CSV batch

Expected columns: `row_id, main_text, country_code, retailer_name, ean, language_code`.


In [ ]:
BATCH_INPUT = PROJECT_ROOT / "inputs" / "products.csv"
BATCH_OUTPUT = PROJECT_ROOT / "data" / "artifacts" / "notebook_batch_summary.csv"


def blank_to_none(value: str | None) -> str | None:
    value = (value or "").strip()
    return value or None


def run_csv_batch(input_path: Path, output_path: Path, feature_set: str) -> list[dict]:
    if not input_path.is_file():
        raise FileNotFoundError(f"{input_path} does not exist")
    with input_path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    summaries = []
    for index, row in enumerate(rows, start=1):
        product = {
            "row_id": blank_to_none(row.get("row_id")) or f"ROW-{index:05d}",
            "main_text": blank_to_none(row.get("main_text")),
            "country_code": blank_to_none(row.get("country_code")),
            "retailer_name": blank_to_none(row.get("retailer_name")),
            "ean": blank_to_none(row.get("ean")),
            "language_code": blank_to_none(row.get("language_code")),
        }
        if not product["main_text"] or not product["country_code"]:
            raise ValueError(f"Row {index} is missing main_text or country_code")
        summary = summarize_result(run_product(product, feature_set))
        summaries.append({
            "row_id": summary["row_id"],
            "job_status": summary["job_status"],
            "coding_ready": summary["coding_ready"],
            "primary_url": summary["primary_url"],
            "serpapi_requests_used": summary["serpapi_requests_used"],
            "candidate_urls_admitted": summary["candidate_urls_admitted"],
            "candidate_investigations_completed": summary["candidate_investigations_completed"],
            "candidate_investigations_failed": summary["candidate_investigations_failed"],
            "selection_scope": summary["selection_scope"],
            "url_decision_status": summary["url_decision_status"],
            "strict_primary_url_accepted": summary["strict_primary_url_accepted"],
            "acceptance_reasons": "|".join(summary["acceptance_reasons"]),
            "coverage": summary["coverage"],
            "missing_features": "|".join(summary["missing_features"]),
            "host_artifact_dir": summary["host_artifact_dir"],
        })
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(summaries[0]) if summaries else ["row_id"])
        writer.writeheader()
        writer.writerows(summaries)
    return summaries


# batch_results = run_csv_batch(BATCH_INPUT, BATCH_OUTPUT, FEATURE_SET)
# pprint(batch_results[:5])
